# Knowledge representation on medical dataset E3C using fine tuned BERT model


## 1. Introduzione e Obiettivo del Task
L'obiettivo di questo progetto è esplorare tecniche avanzate di **Knowledge Representation** applicate al dominio clinico. Nello specifico, il task affrontato è il **Named Entity Recognition (NER)**, formulato computazionalmente come un problema di *Token Classification*. 

Lo scopo del sistema è analizzare referti medici testuali non strutturati ed estrarre automaticamente i concetti di interesse clinico, identificando l'esatta posizione di patologie, sindromi e sintomi (etichettati nel sistema come `CLINENTITY`).

## 2. Il Dataset: European Clinical Case Corpus (E3C)
I dati utilizzati provengono dall'**E3C (European Clinical Case Corpus)**, specificamente dal *Layer 1* (Gold Standard), che contiene annotazioni cliniche validate manualmente da esperti del settore. 

Lavorare con testi medici in italiano presenta una sfida notevole (problema del *Low-Resource Domain*). Il dataset italiano del Layer 1 è costituito da circa 80 documenti. Come dimostreremo, questa scarsità di dati rende il modello addestrato non al massimo dell'ottimalità, causando fenomeni di sbilanciamento delle classi (collasso sull'etichetta `O` - Outside) specialmente nel caso di *Few-Shot Learning*.

## 3. Metodologia ed Esperimenti
Per analizzare e superare questi limiti strutturali, il progetto si articola in una serie di esperimenti a complessità crescente:

* **Baseline Monolingua (BERT Italiano):** Fine-tuning del modello `dbmdz/bert-base-italian-cased` sui soli dati italiani, valutando l'impatto della dimensione del dataset tramite esperimenti *One-Shot* (1 documento), *Few-Shot* (10 documenti) e *Full-Shot* (~80 documenti).
* **Cross-Lingual Transfer Learning:** Per mitigare la scarsità di dati italiani, è stato costruito un pool di addestramento multilingue, fondendo le annotazioni E3C in Italiano, Inglese, Spagnolo, Francese e Basco.
* **Architetture Multilingua (mBERT e XLM-RoBERTa):** Fine-tuning di modelli nativamente multilingua (`bert-base-multilingual-cased` e `xlm-roberta-base`) per dimostrare come i *Word Embeddings* permettano di trasferire la conoscenza semantica delle patologie tra lingue diverse, migliorando nettamente la metrica F1-Score.


# Conversione dati di E3C

I documenti presenti all'interno del dataset di riferimento sono in formato *XML* tuttavia la notazione non è quella classica di riferimento bensì quella *XMI* (XML Metadata Interchange), file di questo tipo provengono da WebAnno ovvero una piattaforma accademica utilizzata per l'annotazione linguistica di testi. I token vengono identificati da ID univoci e su di essi vengono definite le relazioni e le posizioni tra loro nel testo grezzo ovvero, quello contenuto nel tag `Sofa` (Subject of Analisys) presente alla fine del file.

é stata necessaria quindi la creazione di due script che portassero i nostri documenti in formato JSON ovvero `generate_dataset_json.py` e `generate_it_dataset_json.py`. 

Eccone il funzionamento:



In [ ]:
from scripts.generate_it_dataset_json import parse_xmi_e3c
import os

XML_file = "data/raw/E3C-Corpus-2.0.0/data_annotation/Italian/layer1/IT100002.xml"

json_element = parse_xmi_e3c(XML_file)
print(json_element)


Il caso multilingua funziona nella medesima maniera.

In [ ]:
from scripts.generate_dataset_json import parse_xmi_e3c
import os 

XML_file = "data/raw/E3C-Corpus-2.0.0/data_annotation/English/layer1/EN100017.xml"

json_element = parse_xmi_e3c(XML_file)
print(json_element)

Prima di poter dare il via alla procedura di training è necessario fornire al modello BERT i token in formato BIO.

La procedura è contenuta all'interno dello script di training `Train_model.py` nello specifico all'interno della funzione `make_dataset`, tramite questa prendiamo il tokenizzatore facente riferimento al modello BERT selezionato e dopo aver tokenizzato il testo del documento, effettiamo il *BIO tagging* in modo tale da poter dare i token del formato opportuno per la *NER*, convertiamo infine le stringhe in token_ID.

In [1]:
from scripts.train_model import make_dataset

file_json = "data/processed/ita/dataset_train_1_shot.json"
dataset, BIO_token = make_dataset(file_json)
print(BIO_token)

/opt/anaconda3/envs/provareq/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[('[CLS]', 'SPECIAL'), ('G', 'O'), ('.', 'O'), ('b', 'O'), ('.', 'O'), (',', 'O'), ('11', 'O'), ('anni', 'O'), (',', 'O'), ('è', 'O'), ('gi', 'O'), ('##unto', 'O'), ('alla', 'O'), ('nostra', 'O'), ('oss', 'O'), ('##er', 'O'), ('##vazione', 'O'), ('per', 'O'), ('una', 'O'), ('gross', 'O'), ('##a', 'O'), ('tu', 'O'), ('##me', 'O'), ('##fa', 'O'), ('##zione', 'O'), ('della', 'O'), ('cos', 'O'), ('##cia', 'O'), ('dx', 'O'), ('e', 'O'), (',', 'O'), ('dopo', 'O'), ('bio', 'O'), ('##psia', 'O'), ('e', 'O'), ('sta', 'O'), ('##ging', 'O'), (',', 'O'), ('è', 'O'), ('stata', 'O'), ('posta', 'O'), ('dia', 'O'), ('##gno', 'O'), ('##si', 'O'), ('di', 'O'), ('os', 'B-CLINENTITY'), ('##te', 'I-CLINENTITY'), ('##osa', 'I-CLINENTITY'), ('##rco', 'I-CLINENTITY'), ('##ma', 'I-CLINENTITY'), ('del', 'O'), ('fem', 'O'), ('##ore', 'O'), ('destro', 'O'), ('con', 'O'), ('meta', 'B-CLINENTITY'), ('##stas', 'I-CLINENTITY'), ('##i', 'I-CLINENTITY'), ('pol', 'I-CLINENTITY'), ('##mona', 'I-CLINENTITY'), ('##ri', 'I-

## Inferenza sui modelli BERT fine tunati

Nella seguente cella interattiva è implementata una piccola demo dove è possibile provare i modelli addestrati nella fase di training su file di testo medici mai visti o generati da un LLM (gemini, gpt).

Viene data la possibilità all'utente di inserire un testo fittizio oppure, premendo invio prendere un documento medico di default (file txt).

Successivamente il modello BERT selezionato restituisce in output, se presenti, i concetti di interesse clinico (sintomatologia e malattie) con una soglia maggiore o uguale del 80%.

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification


def file_to_string(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            return file.read()
    except FileNotFoundError:
        return "Error: file not found."
    except Exception as e:
        return f"Unexpected error: {e}"


# Set-Up

CARTELLA_MODELLO = "model/multi_roberta_medico_full_shot_early" 
#MODEL_NAME = "dbmdz/bert-base-italian-cased"
#MODEL_NAME = "bert-base-multilingual-cased"
MODEL_NAME = "xlm-roberta-large"

print(f"Caricamento del modello da: {CARTELLA_MODELLO}...")

# tokenizer e modello
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
modello = AutoModelForTokenClassification.from_pretrained(CARTELLA_MODELLO)

# HuggingFace Pipeline
# aggregation_strategy="simple" unisce in automatico i sub-token ("elettro" + "##cardio")
ner_pipeline = pipeline("token-classification", model=modello, tokenizer=tokenizer, aggregation_strategy="simple")

print("\nScrivi un referto medico inventato (o premi Invio per usare l'esempio).")

testo_input = input("\nReferto: ")

    
if not testo_input.strip():
    # predef input file
    testo_input = file_to_string("example/fr_example.txt")
    print(f"Uso l'esempio: {testo_input}")

# Inference
risultati = ner_pipeline(testo_input)

print("\n--- Risultati ---")
if not risultati:
    print("Nessuna entità clinica trovata.")
else:
    for entita in risultati:
        parola = entita['word']
        etichetta = entita['entity_group']
        score = entita['score'] * 100

        if score >= 30:
            print(f" Trovato: '{parola}'")
            print(f"   Tipo: {etichetta} (Score del modello: {score:.1f}%)\n")

Caricamento del modello da: model/multi_roberta_medico_full_shot_early...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


Scrivi un referto medico inventato (o premi Invio per usare l'esempio).
Uso l'esempio: Rapport médical

Le patient, André Martin, âgé de 64 ans, consulte pour une fatigue intense et persistante, une douleur thoracique intermittente, des troubles respiratoires progressifs et des douleurs abdominales récurrentes. Il présente des antécédents d’hypertension artérielle, d’insuffisance cardiaque chronique, de diabète de type 2, d’insuffisance rénale modérée et d’arthrose lombaire. Depuis plusieurs semaines, il décrit une aggravation de l’essoufflement à l’effort, des épisodes de palpitations, un gonflement des membres inférieurs, ainsi qu’une prise de poids récente associée à une rétention hydrique. Il rapporte également une toux productive avec expectorations blanchâtres, des sifflements respiratoires nocturnes et une sensation d’oppression thoracique.

Sur le plan digestif, le patient mentionne des brûlures rétro-sternales, des douleurs épigastriques après les repas, des nausées occasionn

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


#Plotting
sns.set_theme(style="whitegrid", font_scale=1.1)

esperimenti = [
    "One-Shot\n(Italiano)", 
    "Few-Shot (10)\n(Italiano)", 
    "Full-Shot\n(Italiano)", 
    "mBERT\n(Multilingue)", 
    "XLM-RoBERTa\n(Multilingue)"
]

f1_scores = [0.00, 0.46, 0.60, 0.64, 0.67]


colori = ['#e74c3c', '#3498db', '#2980b9', '#f39c12', '#27ae60']


plt.figure(figsize=(11, 6)) 
bars = plt.bar(esperimenti, f1_scores, color=colori)

plt.ylim(0, 0.8) 
plt.ylabel("Metrica F1-Score", fontsize=12, fontweight='bold')
plt.title("Evoluzione delle Performance NER Clinico (Knowledge Representation)", fontsize=16, fontweight='bold', pad=20)

for bar in bars:
    altezza = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, altezza + 0.01, 
             f"{altezza:.2f}", 
             ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.plot(esperimenti, f1_scores, color='gray', linestyle='--', marker='o', alpha=0.5)

plt.tight_layout()
nome_file = 'grafico_risultati_esame.png'
plt.savefig(nome_file, dpi=300) 
print(f"Grafico salvato con successo come: {nome_file}")

modello migliore BErt su diversi training di dimensione variabile
vedere se italiano full shot generalizza bene il multilingua (matrice con i diversi score: training Score)
epoche fissate 100, 
relazione sul notebook 